In [1]:
import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.14.4
Folder  : P03-workbench
  numpy          ok
  pandas         ok
  sklearn        ok

All good. You can carry on to Step 1.


In [2]:
import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> ..\data\delivery_times.csv


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

orders = pd.read_csv(DATA)

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train:", len(X_train), " test:", len(X_test))

train: 480  test: 120


In [4]:
from sklearn.metrics import mean_absolute_error

def score_both_ways(model, name):
    """Train the model, then score it on train and on test."""
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(f"{name:<26} train {train_mae:5.2f}   "
          f"test {test_mae:5.2f}   gap {gap:5.2f}")
    return {"name": name, "train": train_mae,
            "test": test_mae, "gap": gap}

print("helper ready")

helper ready


In [5]:
from sklearn.linear_model import LinearRegression

linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression           train  2.04   test  1.92   gap -0.11


In [6]:
from sklearn.tree import DecisionTreeRegressor

wild_tree = score_both_ways(
    DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)"
)

DecisionTree (no limit)    train  0.00   test  3.43   gap  3.43


In [7]:
small_tree = score_both_ways(
    DecisionTreeRegressor(max_depth=4, random_state=42),
    "DecisionTree (depth 4)",
)

DecisionTree (depth 4)     train  3.66   test  4.23   gap  0.57


In [8]:
from sklearn.ensemble import RandomForestRegressor

forest = score_both_ways(
    RandomForestRegressor(n_estimators=50, random_state=42),
    "RandomForest (50 trees)",
)

RandomForest (50 trees)    train  1.00   test  2.39   gap  1.39


In [9]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest])
results = results.round(2).sort_values("test")

print(results.to_string(index=False))
print()
best = results.iloc[0]
print(f"Lowest test MAE: {best['name']} at {best['test']:.2f} minutes")

                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.39
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57

Lowest test MAE: LinearRegression at 1.92 minutes


In [10]:
from sklearn.model_selection import cross_val_score

def cross_validate(model, name):
    """Average MAE over 5 different train/test splits."""
    scores = -cross_val_score(
        model, X, y, cv=5, scoring="neg_mean_absolute_error"
    )
    print(f"{name:<26} MAE {scores.mean():5.2f} "
          f"(+/- {scores.std():4.2f})")
    return scores.mean()

cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(
    DecisionTreeRegressor(max_depth=4, random_state=42),
    "DecisionTree (depth 4)")
cv_forest = cross_validate(
    RandomForestRegressor(n_estimators=50, random_state=42),
    "RandomForest (50 trees)")

LinearRegression           MAE  2.03 (+/- 0.16)
DecisionTree (depth 4)     MAE  4.57 (+/- 0.18)
RandomForest (50 trees)    MAE  2.69 (+/- 0.40)


In [11]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    mae = -cross_val_score(
        tree, X, y, cv=5, scoring="neg_mean_absolute_error"
    ).mean()
    scores_by_depth[depth] = round(mae, 3)
    print(f"max_depth={str(depth):<5} MAE {mae:5.2f}")

best_depth = min(scores_by_depth, key=scores_by_depth.get)
print()
print("best depth:", best_depth)

max_depth=2     MAE  5.86
max_depth=3     MAE  4.85
max_depth=4     MAE  4.57
max_depth=6     MAE  3.64
max_depth=8     MAE  3.39
max_depth=None  MAE  3.48

best depth: 8


In [13]:
print("Cross-validated MAE, best first:")
for name, mae in sorted(
    {"LinearRegression": cv_linear,
     "DecisionTree(4)": cv_tree,
     "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
):
    print(f"  {name:<20} {mae:5.2f}")

print()
print("Our choice for the delivery service: LinearRegression.")
print("Reasons:")
print("  1. It has the lowest cross-validated error here.")
print("  2. It trains in milliseconds and needs almost no memory.")
print("  3. We can read its coefficients and explain a prediction")
print("     to a restaurant owner. Nobody can read 50 trees.")
print()
print("Be careful with reason 1. This dataset really was built")
print("from a straight line, so a straight line wins. On messier")
print("real data the forest usually wins instead and then you")
print("must decide what its extra accuracy is worth, against")
print("reasons 2 and 3, which do not change.")

Cross-validated MAE, best first:
  LinearRegression      2.03
  RandomForest(50)      2.69
  DecisionTree(4)       4.57

Our choice for the delivery service: LinearRegression.
Reasons:
  1. It has the lowest cross-validated error here.
  2. It trains in milliseconds and needs almost no memory.
  3. We can read its coefficients and explain a prediction
     to a restaurant owner. Nobody can read 50 trees.

Be careful with reason 1. This dataset really was built
from a straight line, so a straight line wins. On messier
real data the forest usually wins instead and then you
must decide what its extra accuracy is worth, against
reasons 2 and 3, which do not change.


---

# Your turn

The walkthrough above is finished. Now you write some code.

There are **3 tasks**. Each one is small. Each one has a hint.
Do them in order.

Where you see `# TODO`, replace that line with your own code. Do not delete
the variable name on the left of the `=` sign --- the self-check at the
bottom looks for exactly that name.

When you have tried all three, run the **self-check** cell at the end. It
prints a table telling you which tasks are correct. You can run it as many
times as you like.

### Task T1 --- Measure overfitting yourself


    Train a `DecisionTreeRegressor` with `max_depth=12` and
    `random_state=42`, and work out its train/test gap by hand.

    Store the three numbers in `T1_train_mae`, `T1_test_mae` and `T1_gap`.

    `T1_gap` must be the test MAE **minus** the train MAE.


> **Hint.** You may not use `score_both_ways` for this one -- write the three lines out, so you can see what the helper was doing. Fit the tree, then call `mean_absolute_error` twice: once with `y_train` and `X_train`, once with `y_test` and `X_test`.

In [14]:
deep_tree = DecisionTreeRegressor(max_depth=12, random_state=42)

deep_tree.fit(X_train, y_train)
T1_train_mae = None
T1_train_mae= mean_absolute_error(y_train, deep_tree.predict(X_train))

T1_test_mae = None
T1_test_mae=mean_absolute_error(y_test, deep_tree.predict(X_test))

T1_gap = None
T1_gap= T1_test_mae - T1_train_mae
print(f"train {T1_train_mae}  test {T1_test_mae}  gap {T1_gap}")

train 0.04695601851851851  test 3.397986111111111  gap 3.3510300925925924


In [15]:
T2_depths = [2, 4, 6, None]
T2_scores = {}
for depth in T2_depths:
    forest = RandomForestRegressor(
        n_estimators=50,
        max_depth=depth,
        random_state=42
    )
    scores = -cross_val_score(
        forest,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error"
    )
    
    T2_scores[depth] = scores.mean()

T2_best_depth = None
T2_best_depth=min(T2_scores, key= T2_scores.get)
print(T2_scores)
print("best:", T2_best_depth)

{2: np.float64(5.422545124201891), 4: np.float64(3.5515271170143676), 6: np.float64(2.830879533681162), None: np.float64(2.7500833333333317)}
best: None


In [16]:
def compare(models):
    results= {}
    for name, m in models.items():
        m.fit(X_train, y_train)
        mae= mean_absolute_error(y_test, m.predict(X_test))
        results[name]=mae

    return results
T3_table = compare({
    "linear": LinearRegression(),
    "tree4": DecisionTreeRegressor(max_depth=4, random_state=42),
    "forest": RandomForestRegressor(n_estimators=50, random_state=42),
})

print(T3_table)

{'linear': 1.924653079833835, 'tree4': 4.228332168713071, 'forest': 2.391816666666663}


In [ ]:
_results = []
def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | the depth-12 tree is trained', lambda: hasattr(deep_tree, 'tree_'))
_check("T1 | T1_train_mae and T1_test_mae are that tree's two scores", lambda: abs(float(T1_train_mae) - mean_absolute_error(y_train, deep_tree.predict(X_train))) < 0.01 and abs(float(T1_test_mae) - mean_absolute_error(y_test, deep_tree.predict(X_test))) < 0.01)
_check('T1 | T1_gap is test minus train, and it is positive', lambda: abs(float(T1_gap) - (float(T1_test_mae) - float(T1_train_mae))) < 1e-6 and float(T1_gap) > 0)
_check('T2 | T2_scores has one score per depth', lambda: set(T2_scores) == {2, 4, 6, None})
_check('T2 | every score is a plausible MAE in minutes', lambda: all(0 < float(v) < 20 for v in T2_scores.values()))
_check('T2 | T2_best_depth is the depth with the lowest score', lambda: T2_best_depth == min(T2_scores, key=T2_scores.get))
_check('T3 | compare() returns one score per model', lambda: set(T3_table) == {'linear', 'tree4', 'forest'})
_check('T3 | the linear score matches the walkthrough', lambda: abs(float(T3_table['linear']) - linear['test']) < 0.01)

print("==================================================================")
print("SELF-CHECK   Practical 03 --- Choosing a Model Honestly")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")
print("")

SELF-CHECK   Practical 03 --- Choosing a Model Honestly
  [PASS]  T1 | the depth-12 tree is trained
  [PASS]  T1 | T1_train_mae and T1_test_mae are that tree's two scores
  [PASS]  T1 | T1_gap is test minus train, and it is positive
  [PASS]  T2 | T2_scores has one score per depth
  [PASS]  T2 | every score is a plausible MAE in minutes
  [PASS]  T2 | T2_best_depth is the depth with the lowest score
  [PASS]  T3 | compare() returns one score per model
  [PASS]  T3 | the linear score matches the walkthrough
------------------------------------------------------------------
  8 of 8 checks passed
Well done. Save the notebook and submit it.


    ---

    ## What to submit

    1. This notebook, with every cell run and its output visible.
2. In a markdown cell at the end, three sentences: which model you would put in front of real customers, why, and what you gave up by choosing it.

    Name your file `PXX_<your-roll-number>.ipynb` before you upload it.

    ### How this practical is marked

    | What is marked | Marks |
    |---|---|
    | Walkthrough run end to end, outputs visible | 3 |
| Task T1 --- overfitting measured as a gap | 2 |
| Task T2 --- best setting found by cross-validation | 3 |
| Task T3 --- a reusable comparison function | 2 |
    | **Total** | **10** |

    ---

    ## Read more

    - scikit-learn --- Underfitting vs overfitting --- <https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html>
- scikit-learn --- Cross-validation --- <https://scikit-learn.org/stable/modules/cross_validation.html>
- scikit-learn --- Decision trees --- <https://scikit-learn.org/stable/modules/tree.html>
- scikit-learn --- Ensemble methods --- <https://scikit-learn.org/stable/modules/ensemble.html>

I would put the Linear Regression model in front of real customers because it achieved the lowest test MAE, meaning it made the most accurate predictions among the models tested. I chose it because it performed better than both the Decision Tree and Random Forest while also being simple and easy to interpret. By choosing Linear Regression, I gave up the ability to capture more complex non-linear relationships that tree-based models can model.